# OneMove Intelligence: Demand Forecasting EDA

This notebook explores spatial and temporal demand patterns to build the baseline demand forecasting model for the dispatch optimizer.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
plt.style.use('seaborn-darkgrid')
sns.set_context('notebook')

## 1. Load Synthetic Demo Data
We utilize the `data/demo_exports` to simulate our production environment.

In [ ]:
import os

data_path = '../data/demo_exports/orders.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f'Loaded {len(df)} orders.')
else:
    print('Generating synthetic dataframe for EDA...')
    dates = pd.date_range(start='2026-05-01', periods=1000, freq='H')
    df = pd.DataFrame({
        'created_at': dates,
        'service_type': np.random.choice(['ride', 'eats', 'grocery', 'courier'], 1000, p=[0.4, 0.3, 0.2, 0.1]),
        'zone_id': np.random.choice(['manhattan', 'brooklyn', 'queens'], 1000),
        'fare_amount': np.random.normal(25, 10, 1000)
    })

df['created_at'] = pd.to_datetime(df['created_at'])
df['hour'] = df['created_at'].dt.hour
df['day_of_week'] = df['created_at'].dt.dayofweek
df.head()

## 2. Temporal Demand Analysis
We observe clear diurnal patterns, with peaks during morning commutes (Rides) and evening dinner hours (Eats).

In [ ]:
hourly_demand = df.groupby(['hour', 'service_type']).size().unstack().fillna(0)
hourly_demand.plot(figsize=(12, 6), linewidth=2)
plt.title('Hourly Order Volume by Service Type')
plt.xlabel('Hour of Day')
plt.ylabel('Order Count')
plt.legend(title='Service')
plt.show()

## 3. Spatial Hotspots
Demand is highly concentrated in specific pickup zones, requiring proactive partner positioning.

In [ ]:
zone_demand = df.groupby('zone_id')['fare_amount'].sum().sort_values(ascending=False)
zone_demand.plot(kind='bar', figsize=(10, 5), color='teal')
plt.title('Total GMV by Zone')
plt.ylabel('Gross Merchandise Value ($)')
plt.xticks(rotation=45)
plt.show()

## 4. Forecasting Model Baseline
We implement an XGBoost baseline for zone-level demand prediction.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

# Feature engineering
features = pd.get_dummies(df[['hour', 'day_of_week', 'service_type', 'zone_id']])
target = df['fare_amount'] # Proxy for demand intensity

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print(f'Baseline MAE: ${mae:.2f}')